In [ ]:
import argparse
import json
import os
import io
import time
from typing import Any, List

import cv2
import numpy as np
from PIL import Image
import ollama

# ================= 配置区域 =================
MODEL_NAME = "qwen2.5vl:7b"

VIDEO_FOLDER = "./utterances_eyeroll" 
# VIDEO_FOLDER = "./utterances_L2CS" 

INPUT_DATA_FILE = "sarcasm_data.json"

# 输出的结果文件
OUTPUT_FILE = "./results/7b_视频原_noutterances.jsonl"

# 抽帧数量
TARGET_FRAME_COUNT = 20
# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    # 保留原有的参数结构，以便兼容命令行调用习惯，虽然在这个脚本中主要依赖全局配置
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    return parser.parse_args(args=[])

def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """
    读取视频文件，均匀抽取指定数量的帧，并转为字节流列表
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # 防止视频过短或读取失败
    if total_frames <= 0:
        return []

    # 均匀抽帧步长
    step = max(1, total_frames // target_count)
    frame_images = []
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if count % step == 0 and saved_count < target_count:
            # OpenCV BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 缩放图片以节省显存 (建议长边不超过 512 或 768)
            pil_img.thumbnail((512, 512)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=70)
            frame_images.append(img_byte_arr.getvalue())
            saved_count += 1
            
        count += 1

    cap.release()
    return frame_images

def construct_prompt(data_item: dict) -> str:
    """
    根据 sarcasm_data.json 的结构构建 Prompt
    """
    utterance = data_item.get("utterance", "")
    speaker = data_item.get("speaker", "Unknown")
    context_list = data_item.get("context", [])
    context_speakers = data_item.get("context_speakers", [])
    show = data_item.get("show", "TV Show")

    # 构建对话上下文文本
    context_text = ""
    if context_list:
        for s, u in zip(context_speakers, context_list):
            context_text += f"- {s}: {u}\n"

    # prompt = (
    #         f"Task: Detect sarcasm in the TV show '{show}' by analyzing the **Red Arrow (Gaze Vector)** movement.\n\n"

    #         f"### 1. VISUAL LEGEND\n"
    #         f"- **Red Arrow**: Represents the gaze direction. Focus on the gaze movement\n"
    #         f"- **Sarcasm Signals**: \n"
    #         f"  A. **Eye-Rolling**: Arrow moves **UP** or to the **SIDE** while speaking.\n"
    #         f"  B. **Gaze Aversion**: Arrow suddenly points **AWAY**.\n"
    #         f"  C. **Gaze Fixation**: Arrow stays fixed on one spot for a long time.\n"
    #         f"  D. **Mismatch**: The facial expression/gaze contradicts the positive words.\n\n"

    #         # 如果有 context，这里加上 context 会更有帮助
    #         f"### 2. INPUT DATA\n"
    #         f"- **Speaker**: {speaker}\n"
    #         f"- **Target Utterance**: \"{utterance}\"\n\n"
    #         # f"### Context:\n{context_text}\n" 

    #         f"### 3. STEPS\n"
    #         # f"1. **Trace Arrow**: Simply describe how the Red Arrow moves across all the frames.\n"
    #         f"1. **Trace Gaze**: Simply describe how the gaze moves across all the frames.\n"
    #         f"2. **Check Conflict**: Does this movement suggest 'Eye-Rolling' or 'insincerity' compared to the text?\n\n"

    #         f"### 4. OUTPUT FORMAT (Strict)\n"
    #         f"Gaze Pattern: <Short description of arrow movement>\n"
    #         f"Consistency: <Match or Conflict with text>\n"
    #         f"Final Answer: {{\"sarcasm\": true}} OR {{\"sarcasm\": false}}"
    #     )


#### 原版
    prompt = (
        f"You are an expert in analyzing multimodal social interactions, specifically detecting sarcasm. "
        f"You will be provided with video frames from the TV show '{show}' and the transcript of the conversation.\n\n"
        
        # f"### Context (Previous Conversation):\n{context_text}\n"
        # f"### Target Utterance:\n- {speaker}: \"{utterance}\"\n\n"
        
        f"### Task:\n"
        f"Analyze the speaker's facial expressions, body language, and the context of the conversation. "
        f"Determine if the target utterance is SARCASTIC or NOT SARCASTIC.\n\n"
        
        f"### Response Format:\n"
        f"1. Thinking out loud: Briefly analyze the visual cues (expressions like eye-rolling, smirking, deadpan face) and contextual incongruity (2-3 sentences).\n"
        f"2. Final Answer: Return strictly valid JSON format: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}."
    )


    return prompt


In [5]:
args = parse_args()
video_root = args.video_folder
input_path = args.input_file

print(f"Loading data from {input_path}...")
try:
    with open(input_path, 'r', encoding='utf-8') as f:
        full_dataset = json.load(f)
except FileNotFoundError:
    print(f"Error: Dataset file not found at {input_path}")

# 检查断点续传
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    processed_ids.add(data.get("id"))
                except:
                    pass
print(f"Found {len(processed_ids)} processed items. Resuming...")

# 打开输出文件准备写入
with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
    
    # 遍历数据集 (dataset是字典结构: "key": {data})
    keys = list(full_dataset.keys())
    
    for i, key_id in enumerate(keys):
        item = full_dataset[key_id]
        
        if key_id in processed_ids:
            # print(f"Skipping {key_id}")
            continue

        # --- 步骤 1: 确定视频路径 ---
        # 假设视频文件名就是 key_id 加上 .mp4，例如 "1_60.mp4"
        video_filename = f"{key_id}.mp4" 
        abs_video_path = os.path.abspath(os.path.join(video_root, video_filename))
        
        # 如果找不到，尝试查找 .mkv 或 .avi 等其他后缀 (可选)
        if not os.path.exists(abs_video_path):
            # 备用检查: 有些数据集可能是 1_60.mkv
            abs_video_path = os.path.abspath(os.path.join(video_root, f"{key_id}.mkv"))
        
        if not os.path.exists(abs_video_path):
            print(f"⚠️ Warning: Video file not found for ID {key_id}: {abs_video_path}")
            continue

        print(f"[{i+1}/{len(keys)}] Processing Task: {key_id} | Video: {video_filename}")

        # --- 步骤 2: 抽取视频帧 ---
        frames = extract_frames(abs_video_path, target_count=TARGET_FRAME_COUNT)
        if not frames:
            print(f"❌ Error: No frames extracted for {key_id}")
            continue

        # --- 步骤 3: 构建 Prompt ---
        text_prompt = construct_prompt(item)

        # --- 步骤 4: 调用 VLM ---
        try:
            response = ollama.chat(
                model=MODEL_NAME,
                messages=[{
                    'role': 'user',
                    'content': text_prompt,
                    'images': frames
                }],
                options={
                    'temperature': 0.1,
                    'num_ctx': 30000 # 增加上下文窗口以适应图片token
                }
            )
            
            model_output = response['message']['content']
            
            # --- 步骤 5: 保存结果 ---
            result_record = {
                "id": key_id,
                "model_output": model_output,
                "ground_truth": item.get("sarcasm")
                # "utterance": item.get("utterance")
            }
            
            f_out.write(json.dumps(result_record, ensure_ascii=False) + "\n")
            f_out.flush()
            print(f"✅ Result for {key_id} saved.")
            
        except Exception as e:
            print(f"❌ Error processing {key_id}: {e}")
            time.sleep(2)


Loading data from sarcasm_data.json...
Found 0 processed items. Resuming...
[1/690] Processing Task: 1_60 | Video: 1_60.mp4
✅ Result for 1_60 saved.
[2/690] Processing Task: 1_70 | Video: 1_70.mp4
✅ Result for 1_70 saved.
[3/690] Processing Task: 1_80 | Video: 1_80.mp4
✅ Result for 1_80 saved.
[4/690] Processing Task: 1_90 | Video: 1_90.mp4
✅ Result for 1_90 saved.
[5/690] Processing Task: 1_105 | Video: 1_105.mp4
✅ Result for 1_105 saved.
[6/690] Processing Task: 1_162 | Video: 1_162.mp4
✅ Result for 1_162 saved.
[7/690] Processing Task: 1_175 | Video: 1_175.mp4
✅ Result for 1_175 saved.
[8/690] Processing Task: 1_182 | Video: 1_182.mp4
✅ Result for 1_182 saved.
[9/690] Processing Task: 1_213 | Video: 1_213.mp4
✅ Result for 1_213 saved.
[10/690] Processing Task: 1_276 | Video: 1_276.mp4
✅ Result for 1_276 saved.
[11/690] Processing Task: 1_340 | Video: 1_340.mp4
✅ Result for 1_340 saved.
[12/690] Processing Task: 1_410 | Video: 1_410.mp4
✅ Result for 1_410 saved.
[13/690] Processing T

# 加载分割完的图片，并调用VLM推理

In [8]:
import os
import json
import glob
import io
import time
from PIL import Image
import ollama

# ================= 配置区域 =================
# 模型名称
MODEL_NAME = "qwen2.5vl:32b"

# 关键帧存放的根目录
# 结构假设: KEYFRAME_ROOT/1_60/frame_0.jpg, KEYFRAME_ROOT/1_60/frame_1.jpg ...
KEYFRAME_ROOT = "./keyframe_smooth" 

# 输入的JSON数据文件 (sarcasm_data.json)
INPUT_DATA_FILE = "sarcasm_data.json"

# 输出的结果文件
OUTPUT_FILE = "./results/32b_分割视频gazepromptp.jsonl"

# 图片处理配置
MAX_IMAGE_WIDTH = 720
JPEG_QUALITY = 80
# ===========================================

def load_keyframes_from_dir(task_id):
    """
    从关键帧目录加载已保存的关键帧图片，并转为字节流
    """
    # 拼接路径，例如 ./keyframes_output/1_60
    keyframe_dir = os.path.join(KEYFRAME_ROOT, task_id)
    
    if not os.path.exists(keyframe_dir):
        # 尝试容错：有些系统可能把 ID 中的 _ 替换成了其他字符，或者文件夹结构不同
        # 这里仅作示例，如果路径严格匹配则不需要
        print(f"⚠️ 关键帧目录不存在: {keyframe_dir}")
        return []
    
    # 获取所有 jpg 文件并按名称排序 (确保时间顺序)
    # 假设文件名是按顺序命名的，如 001.jpg, 002.jpg
    jpg_files = sorted(glob.glob(os.path.join(keyframe_dir, "*.jpg")))
    
    if not jpg_files:
        print(f"⚠️ 在 {keyframe_dir} 中未找到 JPG 文件")
        return []
    
    frame_images = []
    
    for jpg_path in jpg_files:
        try:
            with Image.open(jpg_path) as pil_img:
                # 1. 缩放图片以节省显存 (保持宽高比)
                # 如果图片已经很小，可以跳过这一步
                if pil_img.width > MAX_IMAGE_WIDTH:
                    ratio = MAX_IMAGE_WIDTH / float(pil_img.width)
                    new_height = int(float(pil_img.height) * ratio)
                    pil_img = pil_img.resize((MAX_IMAGE_WIDTH, new_height), Image.Resampling.LANCZOS)
                
                # 2. 转为 JPEG 字节流
                img_byte_arr = io.BytesIO()
                pil_img.save(img_byte_arr, format='JPEG', quality=JPEG_QUALITY)
                frame_data = img_byte_arr.getvalue()
                frame_images.append(frame_data)
        except Exception as e:
            print(f"❌ 读取文件失败 {jpg_path}: {e}")
            continue
    
    return frame_images

def construct_prompt(item: dict) -> str:
    """
    构建 Prompt，包含 Gaze Vector 的说明
    """
    utterance = item.get("utterance", "")
    speaker = item.get("speaker", "Unknown")
    show = item.get("show", "TV Show")
    

    # prompt = (
    #         f"Task: Detect sarcasm in the TV show '{show}' by analyzing the **Red Arrow (Gaze Vector)** movement.\n\n"

    #         f"### 1. VISUAL LEGEND\n"
    #         f"- **Red Arrow**: Represents the eye direction (could be the listener's).\n"
    #         f"- **Sarcasm Signals**: \n"
    #         f"  A. **Eye-Rolling**: Arrow moves **UP** or to the **SIDE** while speaking.\n"
    #         f"  B. **Gaze Aversion**: Arrow suddenly points **AWAY** from the listener.\n"
    #         f"  C. **Mismatch**: The facial expression/gaze contradicts the positive words.\n\n"

    #         # 如果有 context，这里加上 context 会更有帮助
    #         f"### 2. INPUT DATA\n"
    #         f"- **Speaker**: {speaker}\n"
    #         f"- **Utterance**: \"{utterance}\"\n\n"
    #         # f"### Context:\n{context_text}\n" 

    #         f"### 3. STEPS\n"
    #         f"1. **Trace Arrow**: Simply describe how the Red Arrow moves across all the frames.\n"
    #         f"2. **Check Conflict**: Does this movement suggest 'Eye-Rolling' or 'insincerity' compared to the text?\n\n"

    #         f"### 4. OUTPUT FORMAT (Strict)\n"
    #         f"Gaze Pattern: <Short description of arrow movement>\n"
    #         f"Consistency: <Match or Conflict with text>\n"
    #         f"Final Answer: {{\"sarcasm\": true}} OR {{\"sarcasm\": false}}"
    #     )

    prompt = (
        f"""
        You are an expert in Multimodal Sarcasm Detection.
        The provided images are a **chronological sequence of keyframes** capturing the speaker's eye movement changes during a conversation.
        
        # Input Data
        - Speaker: [{speaker}] in the TV show "{show}" 
        - Target Utterance: "{utterance}"
        
        # Task
        Analyze the alignment between the speaker's **gaze behavior** (visual channel) and the **utterance** (verbal channel) to detect sarcasm.
        
        # Analysis Steps (Chain of Thought):
        1. **Visual Intent Analysis**: instead of describing coordinates, interpret the *intent* of the gaze movement shown in the keyframes. 
        - Are they maintaining sincere eye contact? 
        - Is there an **eye-roll** (looking up/side dismissively)? 
        - Is there **gaze avoidance** or a **deadpan stare** that contradicts the emotion of the words?
        2. **Incongruity Check**: Compare the visual intent with the literal meaning of the utterance.
        - Sarcasm Rule: Positive Text + Negative/Dismissive Gaze = Sarcasm.
        - Sarcasm Rule: Emotional Text + Deadpan/Bored Gaze = Sarcasm.

        # Output Format
        1. **Reasoning**: A concise analysis (2-3 sentences) focusing strictly on whether the gaze *contradicts* the text.
        2. **Final Decision**: Strictly valid JSON: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}.
        """
    )
    return prompt

def main():
    # 1. 读取原始数据 (sarcasm_data.json 是字典结构)
    print(f"Loading data from {INPUT_DATA_FILE}...")
    try:
        with open(INPUT_DATA_FILE, 'r', encoding='utf-8') as f:
            full_dataset = json.load(f) # 读取整个字典 {"1_60": {...}, "1_70": {...}}
    except FileNotFoundError:
        print(f"❌ Error: Dataset file not found at {INPUT_DATA_FILE}")
        return

    # 2. 断点续传逻辑
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        data = json.loads(line)
                        processed_ids.add(data.get("id"))
                    except:
                        pass
    print(f"🔄 已处理任务数: {len(processed_ids)}")

    # 3. 循环处理
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
        
        # 遍历字典的 items
        keys = list(full_dataset.keys())
        total_tasks = len(keys)
        
        for i, task_id in enumerate(keys):
            item = full_dataset[task_id]
            
            # 跳过已处理
            if task_id in processed_ids:
                continue
                
            print(f"\n[{i+1}/{total_tasks}] 🎬 Processing Task: {task_id}")
            
            # === Step A: 加载本地关键帧 ===
            frames = load_keyframes_from_dir(task_id)
            
            if not frames:
                print(f"⚠️ Task {task_id}: 无可用关键帧，跳过。")
                continue
            
            print(f"   📸 Loaded {len(frames)} frames.")

            # === Step B: 构建 Prompt ===
            text_prompt = construct_prompt(item)

            # === Step C: 调用 VLM 推理 ===
            try:
                response = ollama.chat(
                    model=MODEL_NAME,
                    messages=[{
                        'role': 'user',
                        'content': text_prompt,
                        'images': frames # 传入加载的图片字节流列表
                    }],
                    options={
                        'temperature': 0.1,
                        'num_ctx': 15000,
                        'num_predict': 512 # 限制输出长度
                    }
                )
                
                model_output = response['message']['content']
                
                # === Step D: 保存结果 ===
                result_record = {
                    "id": task_id,
                    "model_output": model_output,
                    "ground_truth": item.get("sarcasm"),
                    "utterance": item.get("utterance"),
                    "frames_count": len(frames)
                }
                
                f_out.write(json.dumps(result_record, ensure_ascii=False) + "\n")
                f_out.flush()
                print(f"   ✅ Result saved.")
                
            except Exception as e:
                print(f"   ❌ Inference Error: {e}")
                time.sleep(1) # 稍微暂停，防止报错刷屏

if __name__ == "__main__":
    main()

Loading data from sarcasm_data.json...
🔄 已处理任务数: 113

[113/690] 🎬 Processing Task: 1_4850
   📸 Loaded 25 frames.
   ✅ Result saved.

[114/690] 🎬 Processing Task: 1_4949
   📸 Loaded 25 frames.
   ✅ Result saved.

[115/690] 🎬 Processing Task: 1_4967
   📸 Loaded 25 frames.
   ✅ Result saved.

[117/690] 🎬 Processing Task: 1_5058
   📸 Loaded 25 frames.
   ✅ Result saved.

[118/690] 🎬 Processing Task: 1_5109
   📸 Loaded 19 frames.
   ✅ Result saved.

[119/690] 🎬 Processing Task: 1_5134
   📸 Loaded 25 frames.
   ✅ Result saved.

[120/690] 🎬 Processing Task: 1_5156
   📸 Loaded 16 frames.
   ✅ Result saved.

[121/690] 🎬 Processing Task: 1_5166
   📸 Loaded 24 frames.
   ✅ Result saved.

[122/690] 🎬 Processing Task: 1_5169
   📸 Loaded 8 frames.
   ✅ Result saved.

[123/690] 🎬 Processing Task: 1_5211
   📸 Loaded 25 frames.
   ✅ Result saved.

[124/690] 🎬 Processing Task: 1_5212
   📸 Loaded 25 frames.
   ✅ Result saved.

[125/690] 🎬 Processing Task: 1_5496
   📸 Loaded 25 frames.
   ✅ Result saved.


# 结果评估

In [28]:
import json
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# RESULT_FILE = OUTPUT_FILE
RESULT_FILE = "results/32b_原视频原prompt_2.jsonl"

def extract_prediction(model_output):
    """
    从模型输出文本中提取 sarcasm 的布尔值。
    """
    if not model_output:
        return None
    text = model_output.lower()
    match = re.search(r'["\']sarcasm["\']\s*:\s*(true|false)', text)
    if match:
        result_str = match.group(1)
        return True if result_str == 'true' else False
    return None

def calculate_metrics(file_path):
    y_true = []
    y_pred = []
    parse_errors = 0
    error_cases = []  # 用于存储推理错误的 task

    print(f"正在读取文件: {file_path} ...")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            
            data = json.loads(line)
            ground_truth = data.get("ground_truth")
            model_output = data.get("model_output", "")
            prediction = extract_prediction(model_output)
            
            if prediction is None:
                parse_errors += 1
                continue
            
            y_true.append(ground_truth)
            y_pred.append(prediction)

            # --- 新增：判断是否预测错误并记录 ---
            if prediction != ground_truth:
                error_type = "False Positive" if prediction is True else "False Negative"
                error_cases.append({
                    "id": data.get("id"),
                    "model_output": model_output,
                    "ground_truth": ground_truth,
                    "prediction": prediction,
                    "error_type": error_type
                })

    # 计算指标
    if not y_true:
        print("没有有效数据。")
        return

    # 保存错误案例到 error_cases.json
    with open('error_cases_glm.json', 'w', encoding='utf-8') as f:
        json.dump(error_cases, f, indent=4, ensure_ascii=False)
    print(f"✅ 已提取 {len(error_cases)} 个错误案例并保存至 error_cases.json")

    acc = accuracy_score(y_true, y_pred)
    print("\n" + "="*30)
    print(f"📊 评估结果")
    print("="*30)
    print(f"总样本数: {len(y_true)} | 解析失败数: {parse_errors}")
    print(f"✅ 准确率 (Accuracy): {acc:.4f}")
    print("-" * 30)
    
    print("\n详细分类报告:")
    print(classification_report(y_true, y_pred, target_names=["Not Sarcastic", "Sarcastic"], digits=4))
    
    cm = confusion_matrix(y_true, y_pred)
    print("混淆矩阵:")
    print(f"TN: {cm[0][0]} | FP: {cm[0][1]}")
    print(f"FN: {cm[1][0]} | TP: {cm[1][1]}")

# 执行
calculate_metrics(RESULT_FILE)

正在读取文件: results/32b_原视频原prompt_2.jsonl ...
✅ 已提取 223 个错误案例并保存至 error_cases.json

📊 评估结果
总样本数: 690 | 解析失败数: 0
✅ 准确率 (Accuracy): 0.6768
------------------------------

详细分类报告:
               precision    recall  f1-score   support

Not Sarcastic     0.7585    0.5188    0.6162       345
    Sarcastic     0.6344    0.8348    0.7209       345

     accuracy                         0.6768       690
    macro avg     0.6964    0.6768    0.6685       690
 weighted avg     0.6964    0.6768    0.6685       690

混淆矩阵:
TN: 179 | FP: 166
FN: 57 | TP: 288


# Pipeline 融合

In [ ]:
import argparse
import json
import os
import cv2
import io
import time
import re
from typing import List, Dict, Optional, Any
from PIL import Image
from tqdm import tqdm
import ollama

# ================= 配置区域 =================
MODEL_NAME = "qwen2.5vl:7b"

FOLDER_RAW = "./utterances_final"       # Stream A
FOLDER_GAZE = "./utterances_L2CS"       # Stream B

INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/7b_adaptive_fusion.jsonl"

TARGET_FRAME_COUNT = 25

# === 新增配置: 阈值控制 ===
# 只有当 Stream A 判定为 "非讽刺" (False) 且 confidence >= 此阈值时，才跳过 B
# 范围 1-10。设置为 11 则强制所有 case 都跑 B (即 disable 优化)
SKIP_B_THRESHOLD = 8 

# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input-file", default=INPUT_DATA_FILE)
    parser.add_argument("--output-file", default=OUTPUT_FILE)
    return parser.parse_args(args=[])

def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """读取视频文件，均匀抽取指定数量的帧"""
    if not os.path.exists(video_path):
        if os.path.exists(video_path + ".mp4"):
            video_path += ".mp4"
        else:
            return []

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return []

    step = max(1, total_frames // target_count)
    frame_images = []
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret: break   
        if count % step == 0 and saved_count < target_count:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            pil_img.thumbnail((512, 512)) 
            
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=70)
            frame_images.append(img_byte_arr.getvalue())
            saved_count += 1
        count += 1
    cap.release()
    return frame_images

# ================= 解析辅助函数 =================

def parse_model_json(text: str) -> Dict[str, Any]:
    """
    尝试从模型输出中提取 JSON。
    如果失败，返回默认的 '不确定' 状态，强制触发后续流程。
    """
    default_res = {"sarcasm": False, "confidence": 0, "analysis": text}
    
    try:
        # 1. 尝试直接解析
        return json.loads(text)
    except:
        # 2. 尝试提取代码块 ```json ... ```
        match = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                pass
        
        # 3. 尝试提取大括号内容
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                pass
                
    # 如果都失败了，为了安全起见，将 confidence 设为 0，强制让 Pipeline 进入 Stream B
    print(f"Warning: Failed to parse JSON. Raw text: {text[:50]}...")
    return default_res

# ================= Prompt 构建区 =================

def get_global_prompt_with_score(data_item: dict) -> str:
    """Stream A: 要求输出置信度分数的 Prompt"""
    utterance = data_item.get("utterance", "")
    speaker = data_item.get("speaker", "Unknown")
    context_list = data_item.get("context", [])
    context_speakers = data_item.get("context_speakers", [])
    show = data_item.get("show", "TV Show")

    context_text = ""
    if context_list:
        for s, u in zip(context_speakers, context_list):
            context_text += f"- {s}: {u}\n"

    prompt = (
        f"Analyze this video clip from '{show}'.\n"
        f"### Context:\n{context_text}\n"
        f"### Target Utterance:\n- {speaker}: \"{utterance}\"\n\n"
        f"### Task:\n"
        f"1. Analyze facial expressions, tone, and context.\n"
        f"2. Determine if it is SARCASTIC.\n"
        f"3. Provide a **Confidence Score** (1-10) for your judgment. (10=Certain, 1=Guessing).\n\n"
        f"### STRICT Output Format (JSON only):\n"
        f"{{\n"
        f"  \"analysis\": \"...reasoning...\",\n"
        f"  \"sarcasm\": true/false,\n"
        f"  \"confidence\": int\n"
        f"}}"
    )
    return prompt

def get_gaze_prompt(data_item: dict) -> str:
    """Stream B: Gaze 依然保持文本输出，因为它的 reasoning 需要喂给 Fusion"""
    utterance = data_item.get("utterance", "")
    speaker = data_item.get("speaker", "Unknown")
    
    prompt = (
        f"Task: Detect sarcasm by analyzing the **Red Arrow (Gaze Vector)** movement.\n"
        f"Speaker: {speaker}, Utterance: \"{utterance}\"\n"
        f"Signals: Eye-Rolling, Aversion, Mismatch.\n\n"
        f"Output:\n"
        f"1. Gaze Pattern: Describe movement.\n"
        f"2. Consistency: Support or contradict text?\n"
        f"3. Verdict: Is it sarcastic based on Gaze?"
    )
    return prompt

def get_fusion_prompt(res_a_analysis: str, res_b: str, data_item: dict) -> str:
    utterance = data_item.get("utterance", "")
    
    prompt = (
        f"Arbitrate Sarcasm Detection.\nUtterance: \"{utterance}\"\n\n"
        f"### Analysis A (Context View):\n{res_a_analysis}\n\n"
        f"### Analysis B (Gaze View):\n{res_b}\n\n"
        f"### Rules:\n"
        f"- Report A is generally reliable for literal meaning.\n"
        f"- Report B tends to hallucinate. Only trust B if it describes a VERY distinct sarcastic eye movement.\n"
        f"- Resolve the conflict.\n\n"
        f"### Output JSON:\n"
        f"{{\"reasoning\": \"...\", \"sarcasm\": true/false}}"
    )
    return prompt

# ================= 推理核心 =================

def call_vlm(images: List[bytes], prompt: str, model: str = MODEL_NAME) -> str:
    try:
        message = {'role': 'user', 'content': prompt}
        if images: message['images'] = images

        response = ollama.chat(
            model=model,
            messages=[message],
            options={'temperature': 0.1} 
        )
        return response['message']['content']
    except Exception as e:
        print(f"Error calling VLM: {e}")
        return "{}"

# ================= 主流程 =================

def main():
    args = parse_args()
    
    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    results_file = open(args.output_file, 'w', encoding='utf-8')

    for key, item in tqdm(data.items()):
        video_id = key 
        path_raw = os.path.join(FOLDER_RAW, f"{video_id}.mp4")
        
        # 1. 抽取 Raw 帧 (Stream A 总是要跑的)
        frames_raw = extract_frames(path_raw)
        if not frames_raw:
            print(f"Skipping {video_id}: Raw video not found.")
            continue

        # 2. 运行 Stream A (Global Context)
        prompt_a = get_global_prompt_with_score(item)
        res_a_text = call_vlm(frames_raw, prompt_a)
        
        # 解析 Stream A 的 JSON 结果
        res_a_data = parse_model_json(res_a_text)
        
        # 提取关键指标
        sarcasm_a = res_a_data.get("sarcasm", False)
        confidence_a = res_a_data.get("confidence", 0)
        analysis_a = res_a_data.get("analysis", res_a_text)

        # 3. 条件判断逻辑 (Gatekeeping Logic)
        # 策略: 
        # 如果 A 认为是“非讽刺” 且 置信度很高 (>=8) -> 相信 A，不再浪费算力跑 B。
        # 如果 A 认为是“讽刺” (通常 A 比较保守，如果是讽刺那大概率就是) -> 也可以选择不跑 B，或者为了 Explainability 跑 B。
        # 如果 A “不确定” (Confidence < 8) -> 必须跑 B 来辅助。
        
        need_stream_b = True
        
        # 你的核心逻辑: 如果是 Non-Sarcasm 且很确定，就不需要 Gaze 介入了
        if (not sarcasm_a) and (confidence_a >= SKIP_B_THRESHOLD):
            need_stream_b = False
        
        # 也可以加一条：如果 A 极其确信是讽刺(10分)，也许也不需要 B? 
        # 但通常为了找证据（Evidence），保留 B 是好的。这里暂不加。

        res_b = "Skipped (High confidence in Stream A)"
        res_final = ""
        final_verdict = sarcasm_a # 默认为 A 的判断

        if need_stream_b:
            # 需要 B 介入
            path_gaze = os.path.join(FOLDER_GAZE, f"{video_id}.mp4")
            frames_gaze = extract_frames(path_gaze)
            
            if frames_gaze:
                # 运行 Stream B
                prompt_b = get_gaze_prompt(item)
                res_b = call_vlm(frames_gaze, prompt_b)
                
                # 运行 Fusion (Arbitration)
                prompt_fusion = get_fusion_prompt(analysis_a, res_b, item)
                res_final_text = call_vlm([], prompt_fusion) # 纯文本推理
                
                # 尝试解析最终结果
                res_final_data = parse_model_json(res_final_text)
                final_verdict = res_final_data.get("sarcasm", False) # 最终 verdict
                res_final = res_final_text
            else:
                res_b = "Error: Gaze video not found"
        else:
            # 不需要 B 介入，直接使用 A 的 Reasoning 作为最终结果
            # 为了格式统一，伪造一个 JSON 字符串
            res_final = json.dumps({
                "reasoning": f"Stream A detected Non-Sarcasm with high confidence ({confidence_a}/10). Gaze stream skipped for efficiency.",
                "sarcasm": sarcasm_a
            })

        # 4. 保存结果
        output_obj = {
            "id": key,
            "ground_truth": item.get("sarcasm", None),
            "stream_A_sarcasm": sarcasm_a,
            "stream_A_confidence": confidence_a,
            "stream_A_analysis": analysis_a,
            "stream_B_gaze_raw": res_b,
            "fusion_result": res_final,
            "final_prediction": final_verdict,
            "pipeline_path": "Dual-Stream" if need_stream_b else "Single-Stream-A"
        }
        
        results_file.write(json.dumps(output_obj, ensure_ascii=False) + "\n")
        results_file.flush()

    results_file.close()
    print(f"Done! Results saved to {args.output_file}")

if __name__ == "__main__":
    main()

  1%|          | 4/690 [00:38<1:38:48,  8.64s/it]

Error calling VLM: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


  1%|          | 8/690 [01:26<1:53:12,  9.96s/it]

Error calling VLM: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


 11%|█▏        | 78/690 [10:53<1:22:20,  8.07s/it]

...

Error calling VLM: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


 97%|█████████▋| 666/690 [1:30:15<03:11,  8.00s/it]

Error calling VLM: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed (status code: 500)


100%|██████████| 690/690 [1:33:09<00:00,  8.10s/it]

Done! Results saved to ./results/7b_fusion_2.jsonl


# 先VLM识别gaze，再LLM推理

In [ ]:
import json
import os
import cv2
import io
import argparse
import base64
from typing import List, Dict, Any
from PIL import Image
from tqdm import tqdm
import ollama  
import glob

# ================= 配置区域 =================
# 视觉模型：负责提取Gaze特征
VLM_MODEL_NAME = "qwen2.5vl:7b"
# 语言模型：负责逻辑推理
LLM_MODEL_NAME = "qwen2.5:7b"

# VIDEO_FOLDER = "./utterances_final" 
# VIDEO_FOLDER = "./utterances_L2CS" 
VIDEO_FOLDER = "./utterances_gazelle" 
INPUT_DATA_FILE = "sarcasm_data.json"
OUTPUT_FILE = "./results/7b_VLM+LLM_gazelle.jsonl"

# 抽帧数量 (Qwen2.5-VL处理多图能力较强，但过多会影响速度和显存)
TARGET_FRAME_COUNT = 20

KEYFRAME_ROOT = "/projects/kzh/VIBE_gaze/Sarcasm/keyframe_smooth"
RESIZE_DIM = 720  # 最大宽度 (从 1920 降到 720)
JPEG_QUALITY = 80      
# ===========================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--video-folder", default=VIDEO_FOLDER, help="Path to video folder")
    parser.add_argument("--input-file", default=INPUT_DATA_FILE, help="Path to input json file")
    parser.add_argument("--output-file", default=OUTPUT_FILE, help="Path to output result file")
    return parser.parse_args(args=[])

def load_keyframes_from_dir(task_id: str) -> List[bytes]:
    """
    针对特定路径结构优化的加载函数
    路径示例: /projects/kzh/VIBE_gaze/Sarcasm/keyframe_smooth/{task_id}/key_*.jpg
    """
    # 1. 组合完整目录路径
    keyframe_dir = os.path.join(KEYFRAME_ROOT, str(task_id))
    # print(keyframe_dir)
    
    if not os.path.exists(keyframe_dir):
        print(f"⚠️ 关键帧目录不存在: {keyframe_dir}")
        return []
    
    # 2. 查找 key_ 开头的 jpg 文件
    # 使用通配符以匹配 key_00_frame_0.jpg 等格式
    search_pattern = os.path.join(keyframe_dir, "key_*.jpg")
    jpg_files = sorted(glob.glob(search_pattern))
    
    # 如果 key_ 开头没找到，退而求其次寻找所有 jpg
    if not jpg_files:
        jpg_files = sorted(glob.glob(os.path.join(keyframe_dir, "*.jpg")))
    
    if not jpg_files:
        print(f"⚠️ 在 {keyframe_dir} 中未找到图片。尝试搜索模式: {search_pattern}")
        return []
    
    frame_images = []
    
    for jpg_path in jpg_files:
        try:
            with Image.open(jpg_path) as pil_img:
                # 必须转 RGB，否则 JPEG 保存会失败（特别是如果有索引色或Alpha通道）
                if pil_img.mode != 'RGB':
                    pil_img = pil_img.convert('RGB')
                
                # 缩放至与 extract_frames 一致
                pil_img.thumbnail((RESIZE_DIM, RESIZE_DIM)) 
                
                img_byte_arr = io.BytesIO()
                pil_img.save(img_byte_arr, format='JPEG', quality=JPEG_QUALITY)
                
                frame_images.append(img_byte_arr.getvalue())
        except Exception as e:
            print(f"❌ 处理文件失败 {jpg_path}: {e}")
            continue
    
    print(f"✅ 从 {task_id} 文件夹成功加载了 {len(frame_images)} 张关键帧")
    return frame_images


def extract_frames(video_path: str, target_count: int = TARGET_FRAME_COUNT) -> List[bytes]:
    """
    读取视频文件，均匀抽取指定数量的帧，并转为字节流列表 (适配Ollama输入)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ 无法打开视频: {video_path}")
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    step = max(1, total_frames // target_count)
    frame_bytes_list = []
    
    count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if count % step == 0 and saved_count < target_count:
            # OpenCV BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # 缩放图片以节省显存和Token (推荐长边512或448)
            pil_img.thumbnail((512, 512)) 
            
            # 转为字节流
            img_byte_arr = io.BytesIO()
            pil_img.save(img_byte_arr, format='JPEG', quality=70)
            frame_bytes_list.append(img_byte_arr.getvalue())
            saved_count += 1
            
        count += 1

    cap.release()
    return frame_bytes_list

def stage1_extract_gaze(frames: List[bytes], speaker_name: str) -> str:
    """
    Stage 1: 使用 VLM (Qwen2.5-VL) 提取眼神描述
    """
    if not frames:
        return "No visual information available."

    # 动态构建Prompt，将硬编码的[PENNY]替换为实际说话人
    # prompt = (
    #     f"""
    #     You are a Human Gaze and Analysis Expert.
    #     Your task is to describe the visual behavior of the main speaker [{speaker_name}] in the video frames (less than 3 sentences). 
    #     **The red arrow indicates the gaze direction.**
        
    #     # Input semantics (from symbolic gaze reference)
    #     - Event types: Fixation, Saccade, SmoothPursuit
    #     - Use the exact tokens when referring to event types: "fixation", "saccade", "smooth pursuit" (avoid synonyms like glance, gaze shift, tracking)
    #     - Use qualitative words implied by labels only: duration → brief/short/moderate/long; amplitude → small/medium/large; speed → slow/fast/very fast (pursuit may use steady)

    #     # Style & constraints
    #     - Summarize the most salient gaze patterns of the main speaker [{speaker_name}];
    #     - Maintain temporal continuity across gaze shifts;
    #     - Be purely descriptive and objective. Do NOT interpret *why* they are doing it.;
    #     """
    # )
    #### GazeLLE 版本的prompt
    prompt = (
            f"""
            You are an expert in Gaze Analysis.
            Analyze the video frame-by-frame to track the Green Line emerging from {speaker_name}'s face.

            # Step 1: Temporal Detection
            - Does the Green Line exist continuously? Or does it break/disappear at certain moments?
            - **Line Visible** -> Fixation
            - **Line Gone** -> Saccade/Shift

            # Step 2: Sequence Construction
            - Identify the pattern: Fixation -> Saccade -> Fixation.
            - Describe this dynamic rhythm. Avoid using static words like "steady" or "constant".

            # Final Answer:
            Provide a single, dynamic sentence describing the sequence of gaze events .
            """
        )

    try:
        response = ollama.chat(
            model=VLM_MODEL_NAME,
            messages=[
                {
                    'role': 'user',
                    'content': prompt,
                    'images': frames  # Ollama Python库支持直接传bytes列表
                }
            ],
            options={'temperature': 0.1, 'num_ctx': 30000} 
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"⚠️ VLM Error: {e}")
        return "Error extracting gaze description."

def stage2_infer_sarcasm(utterance: str, gaze_desc: str, context: str, context_speakers: List[str]) -> dict:
    """
    Stage 2: 使用 LLM (Qwen2.5) 进行讽刺推理
    """
    
    prompt = (
        f"""
        You are a pragmatic Communication Analyst.
        Your task is to determine if the target utterance is **SARCASTIC** based on the visual description and conversation context.
        # Target utterance: "{utterance}"
        # Gaze description: "{gaze_desc}"
        # Context: \n{context}
        # Context speakers: {context_speakers}

        # Task:
        1. Analyze the speaker's gaze description, utterance, and the context of the conversation. 
        2. Determine if the target utterance is SARCASTIC or NOT SARCASTIC.

        ### Response Format:
        1. Thinking out loud: Analyze the gaze description, and the incongruity (2-3 sentences).
        2. Final Answer: Return strictly valid JSON format: {{\"sarcasm\": true}} or {{\"sarcasm\": false}}.
        """
    )

    try:
        response = ollama.chat(
            model=LLM_MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.1, 'num_ctx': 30000} 
        )
        content = response['message']['content']
        
        return parse_llm_json(content)
        
    except Exception as e:
        print(f"⚠️ LLM Error: {e}")
        return {"sarcasm": False, "error": str(e), "raw_response": ""}

def parse_llm_json(content: str) -> dict:
    """
    解析 LLM 可能带有 Markdown 格式的 JSON 输出
    """
    try:
        # 尝试直接解析
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    try:
        if "```json" in content:
            json_str = content.split("```json")[1].split("```")[0].strip()
            return json.loads(json_str)
        elif "{" in content and "}" in content:
            start = content.find("{")
            end = content.rfind("}") + 1
            json_str = content[start:end]
            return json.loads(json_str)
    except Exception:
        pass
    
    # 如果解析失败，保留思维链文本，标记解析错误
    return {"sarcasm": False, "parsing_error": True, "raw_response": content}

def main():
    args = parse_args()
    
    # 确保输出目录存在
    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)

    # 读取输入数据
    with open(args.input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 检查已处理的数据（断点续传）
    processed_ids = set()
    if os.path.exists(args.output_file):
        with open(args.output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    res = json.loads(line)
                    # 假设数据有 'id' 字段，如果没有，可以使用 utterance 做 key
                    if 'video_id' in res: 
                        processed_ids.add(res['video_id'])
                except:
                    pass

    print(f"🚀 开始处理 Pipeline: VLM({VLM_MODEL_NAME}) -> LLM({LLM_MODEL_NAME})")
    
    items = data.items() if isinstance(data, dict) else enumerate(data)

    with open(args.output_file, 'a', encoding='utf-8') as out_f:
        for vid_id, item in tqdm(items, desc="Processing Videos"):
            
            # 兼容处理：如果数据是list，item本身就是字典；如果是dict，item是value
            if isinstance(data, list):
                # 尝试从item里找个唯一标识，如果没有就用索引
                current_id = item.get('video_id', str(vid_id))
            else:
                current_id = vid_id
            
            if current_id in processed_ids:
                continue

            video_file = f"{current_id}.mp4" # 假设视频文件名
            video_path = os.path.join(args.video_folder, video_file)
            
            # 0. 准备数据
            speaker = item.get("speaker", "The Speaker")
            utterance = item.get("utterance", "")
            context_list = item.get("context", [])
            context_speakers = item.get("context_speakers", [])
            
            # 格式化上下文
            context_str = ""
            for s, u in zip(context_speakers, context_list):
                context_str += f"- {s}: {u}\n"

            # ================= Stage 1: VLM 提取 Gaze =================
            frames = extract_frames(video_path, target_count=TARGET_FRAME_COUNT)
            # frames = load_keyframes_from_dir(current_id)  
            
            if not frames:
                # 视频读取失败，记录失败结果
                result = {
                    "video_id": current_id,
                    "error": "Video load failed",
                    "sarcasm_prediction": False
                }
            else:
                gaze_description = stage1_extract_gaze(frames, speaker)
                
                # ================= Stage 2: LLM 推理 Sarcasm =================
                llm_result = stage2_infer_sarcasm(utterance, gaze_description, context_str, context_speakers)
                
                # 整合结果
                result = {
                    "video_id": current_id,
                    # "speaker": speaker,
                    # "utterance": utterance,
                    "ground_truth": item.get("sarcasm", None), 
                    "gaze_description": gaze_description,      
                    "prediction_reasoning": llm_result.get("raw_response", llm_result.get("Thinking out loud", "")),
                    "sarcasm_prediction": llm_result.get("sarcasm", False)
                }

            # 写入结果
            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()

    print(f"✅ 处理完成，结果已保存至 {args.output_file}")

if __name__ == "__main__":
    main()

🚀 开始处理 Pipeline: VLM(qwen2.5vl:7b) -> LLM(qwen2.5:7b)


Processing Videos:   0%|          | 0/690 [00:00<?, ?it/s]

Processing Videos:  36%|███▌      | 245/690 [03:28<06:18,  1.17it/s]


KeyboardInterrupt: 

In [ ]:
import json
import os

# ================= 配置 =================
RESULTS_FILE = "/projects/kzh/VIBE_gaze/Sarcasm/results/7b_context原视频原prompt_1.jsonl"  # 你的结果文件路径
ERROR_OUTPUT_FILE = "./results/error_cases.json"      # 错误样本保存路径
# =======================================

def evaluate_results():
    if not os.path.exists(RESULTS_FILE):
        print(f"❌ 文件不存在: {RESULTS_FILE}")
        return

    total = 0
    correct = 0
    
    # 混淆矩阵计数 (用于计算 P/R/F1)
    tp = 0  # 真阳性 (GT=True, Pred=True)
    tn = 0  # 真阴性 (GT=False, Pred=False)
    fp = 0  # 假阳性 (GT=False, Pred=True)
    fn = 0  # 假阴性 (GT=True, Pred=False)

    error_cases = []

    with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            try:
                item = json.loads(line)
                
                gt = item.get("ground_truth")
                pred = item.get("sarcasm_prediction")
                
                # 过滤掉标签无效的数据
                if gt is None:
                    continue

                total += 1
                
                # 判断正确与否
                if gt == pred:
                    correct += 1
                    if gt is True: tp += 1
                    else: tn += 1
                else:
                    # 记录错误样本
                    if pred is True: fp += 1
                    else: fn += 1
                    
                    # 标记错误类型
                    item["error_type"] = "False Positive" if pred else "False Negative"
                    error_cases.append(item)

            except json.JSONDecodeError:
                print(f"⚠️ 跳过无效的 JSON 行")
                continue

    # --- 计算指标 ---
    if total == 0:
        print("没有有效的测试数据。")
        return

    accuracy = (correct / total) * 100
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- 打印结果 ---
    print("\n" + "="*30)
    print(f"📊 评估报告 (Total samples: {total})")
    print("="*30)
    print(f"✅ Accuracy (准确率):  {accuracy:.2f}%")
    print(f"🎯 Precision (精确率): {precision:.4f}")
    print(f"📢 Recall (召回率):    {recall:.4f}")
    print(f"⚖️  F1-Score:         {f1:.4f}")
    print("-" * 30)
    print(f"具体分布:")
    print(f"TP (讽刺且预测对): {tp}")
    print(f"TN (正常且预测对): {tn}")
    print(f"FP (误判为讽刺):   {fp}")
    print(f"FN (漏判讽刺):     {fn}")
    print("="*30)

    # --- 保存错误样本 ---
    with open(ERROR_OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(error_cases, f, indent=4, ensure_ascii=False)
    
    print(f"\n📝 发现 {len(error_cases)} 个错误样本，已保存至: {ERROR_OUTPUT_FILE}")

if __name__ == "__main__":
    evaluate_results()


📊 评估报告 (Total samples: 690)
✅ Accuracy (准确率):  0.00%
🎯 Precision (精确率): 0.0000
📢 Recall (召回率):    0.0000
⚖️  F1-Score:         0.0000
------------------------------
具体分布:
TP (讽刺且预测对): 0
TN (正常且预测对): 0
FP (误判为讽刺):   0
FN (漏判讽刺):     690

📝 发现 690 个错误样本，已保存至: ./results/error_cases.json
